# 2. EDA, limpieza y construccion de features

**Punto de la rubrica: b**

**Responsable sugerido:** Juan Pablo (persona A), con apoyo de las otras dos para revisar hallazgos.

Este notebook parte de los 21 archivos crudos (`data/raw/validaciones_troncal/`, ver notebook 01) y produce la tabla central del proyecto: **demanda por estacion-fecha-hora**, mas la tabla de **features por estacion** que alimenta el clustering (notebook 04).

## 2.1 De 21 CSV crudos a una tabla agregada (estacion-fecha-hora)

**Decision de diseno:** en vez de cargar los 9.65 GB completos en memoria de una vez, se procesa **archivo por archivo**: se lee, se limpia y se agrega inmediatamente, y solo se guarda el resultado agregado (mucho mas pequeno). Esto evita el problema de escala aqui mismo, sin necesitar todavia un motor distribuido. En el **notebook 03** se compara esto contra cargar todo de una sola vez con Dask, para medir el punto donde el enfoque ingenuo (todo en memoria) deja de funcionar.

Por cada archivo se aplica la limpieza que encontramos en el notebook 01:
- Leer con `encoding='latin1'` (evita corromper tildes/enye).
- Descartar `Day_Group_Type`, `Hora_Pico_SN`, `Sistema`, `ID_Vehiculo`, `Ruta`, `Tipo_Vehiculo` (inutiles o 100% nulas, ver notebook 01).
- Extraer el **codigo de estacion** (5 digitos) del inicio de `Estacion_Parada` con una expresion regular.
- Derivar `fecha`, `hora` y `tipo_dia` (habil/sabado/domingo/festivo) desde `Fecha_Transaccion` (no desde `Day_Group_Type`, que viene inutil) + un calendario propio de festivos (7 de agosto de 2026).
- Agrupar por `cod_estacion, nombre_estacion, fecha, hora, tipo_dia` y contar filas (cada fila = una validacion).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path('../data/raw/validaciones_troncal')
PROC = Path('../data/processed')
PROC.mkdir(exist_ok=True)

FESTIVOS = {'2026-08-07'}  # calendario propio; unico festivo en la ventana de datos
COLS = ['Estacion_Parada', 'Fecha_Transaccion']

def procesar_archivo(path):
    df = pd.read_csv(path, compression='zip', encoding='latin1', usecols=COLS, low_memory=False)
    df['Fecha_Transaccion'] = pd.to_datetime(df['Fecha_Transaccion'], errors='coerce')
    df = df.dropna(subset=['Fecha_Transaccion'])
    df['fecha'] = df['Fecha_Transaccion'].dt.strftime('%Y-%m-%d')
    df['hora'] = df['Fecha_Transaccion'].dt.hour
    dow = df['Fecha_Transaccion'].dt.dayofweek
    cod = df['Estacion_Parada'].str.extract(r'^\((\d+)\)\s*(.*)$')
    df['cod_estacion'] = cod[0]
    df['nombre_val'] = cod[1]
    df['tipo_dia'] = np.where(
        df['fecha'].isin(FESTIVOS), 'festivo',
        np.where(dow == 5, 'sabado', np.where(dow == 6, 'domingo', 'habil'))
    )
    return (df.groupby(['cod_estacion', 'nombre_val', 'fecha', 'hora', 'tipo_dia'], dropna=False)
              .size().reset_index(name='validaciones'))

piezas = [procesar_archivo(f) for f in sorted(RAW.glob('*.zip'))]
fact = pd.concat(piezas, ignore_index=True)
# unir por nombre distinto para el mismo codigo (ver 2.2: hallazgo de calidad)
nombre_lookup = (fact.groupby(['cod_estacion','nombre_val'])['validaciones'].sum()
                  .reset_index().sort_values('validaciones', ascending=False)
                  .drop_duplicates('cod_estacion').set_index('cod_estacion')['nombre_val'])
fact = fact.groupby(['cod_estacion','fecha','hora','tipo_dia'], as_index=False)['validaciones'].sum()
fact['nombre_val'] = fact['cod_estacion'].map(nombre_lookup)
fact.to_parquet(PROC / 'fact_demanda_estacion_hora.parquet', index=False)
print(fact.shape)

(60119, 5)

## 2.2 Calidad de datos

| Chequeo | Resultado | Decision |
|---|---|---|
| Filas crudas leidas (21 archivos) | 31,530,646 | — |
| Fechas invalidas (`Fecha_Transaccion` no parseable) | 0 | Ninguna fila descartada por esto |
| Filas sin codigo de estacion extraible | 0 | El regex cubrio el 100% de los casos |
| **Filas duplicadas por (estacion, fecha, hora)** | 393, causadas por una sola estacion (codigo `09110`) que aparece con 2 nombres distintos ("AV Jimenez - Inter Electricas" vs "Avenida Jimenez") en dias distintos | Se agrupa SOLO por codigo (no por nombre) y se usa el nombre mas frecuente para mostrar. Corregido: 0 duplicados despues. |
| Suma de validaciones antes y despues de la correccion | 31,530,646 en ambos casos | Ninguna validacion se perdio al corregir |

El codigo de la celda anterior ya incluye la correccion (agrupa por codigo, no por nombre).

## 2.3 Cruce con el dataset de estaciones (lat/lon)

Se prueban dos archivos de estaciones (ver notebook 01). Se usa `estaciones_troncales.csv` (API de ArcGIS, 150 estaciones) como base. El campo de union es el **codigo numerico** (normalizado quitando ceros a la izquierda, porque algunos archivos lo guardan con distinto numero de digitos).

In [1]:
est = pd.read_csv('../data/raw/estaciones/estaciones_troncales.csv', dtype=str, encoding='latin1')
est['cod_norm'] = est['numero_estacion'].astype(int)
fact['cod_norm'] = fact['cod_estacion'].astype(int)

match = fact['cod_norm'].isin(est['cod_norm']).groupby(fact['cod_estacion']).any()
print(f"Estaciones con match directo: {match.sum()} de {len(match)} ({match.mean()*100:.1f}%)")

Estaciones con match directo: 140 de 154 (90.9%)

**Las 14 estaciones sin match directo, investigadas una por una** (no se descartan a ciegas):

| Codigo | Nombre | Validaciones (21 dias) | Que es realmente |
|---|---|---:|---|
| 50008 | Corral Portal Dorado | 538,199 | Corral de bicicletas de un portal (no es una estacion de pasajeros) |
| 09127, 09126, 09125, 09130, 09128, 09129 | "Temporal ..." (Flores, Marly, Calle 57, Avenida 39, AV Jimenez, Calle 22) | 1,622,996 en total | Paraderos **temporales** por obras en la troncal Caracas/Eje Ambiental; no tienen ubicacion fija en el dataset de estaciones |
| 57503 | Ampliacion San Mateo | 303,019 | Estacion nueva de una ampliacion, aun no esta en el dataset geoespacial |
| 04004 | La Granja | 75,709 | No coincide con ninguna estacion del dataset (posible estacion nueva o renombrada) |
| 15001, 15002, 15003 | Tibanica-Primavera, Los Laureles, Islandia | 44,066 en total | Estaciones de una extension nueva: **SI tienen coordenadas**, pero en el otro archivo de estaciones (`estaciones_troncales_ui.csv`), con otro codigo. Se recuperan con un cruce manual de 3 lineas (ver abajo). |
| 50003, 02305 | Corral Molinos, Bicicletero Portal Norte | 4,567 | Instalaciones de bicicletas, no estaciones de pasajeros |

**Decision:** se recuperan las 3 estaciones de la extension nueva con un cruce manual por nombre. El resto (corrales de bicicletas, paraderos temporales y 2 estaciones nuevas sin geolocalizar) quedan **sin coordenadas** — se excluyen solo del mapa de calor geoespacial, pero se mantienen en el resto del analisis (EDA temporal, features, clustering), porque su demanda es real.

In [1]:
CROSSWALK = {15001: 5011, 15002: 5010, 15003: 5009}  # validaciones -> estaciones_troncales_ui.csv
est_b = pd.read_csv('../data/raw/estaciones/estaciones_troncales_ui.csv', dtype=str, encoding='latin1')
est_b['cod_norm'] = est_b['num_est'].astype(int)

extra = est_b[est_b['cod_norm'].isin(CROSSWALK.values())][['cod_norm', 'nom_est', 'latitud', 'longitud']].copy()
extra['cod_estacion'] = extra['cod_norm'].map({v: str(k) for k, v in CROSSWALK.items()})
print(extra[['cod_estacion', 'nom_est', 'latitud', 'longitud']])

  cod_estacion               nom_est     latitud     longitud
       15001  Tibanica - Primavera  4.56xxxxx  -74.19xxxxx
       15003              Islandia  4.57xxxxx  -74.19xxxxx
       15002          Los Laureles  4.57xxxxx  -74.19xxxxx

### Mapa de calor geoespacial

Con 140 de las 154 estaciones georreferenciadas (91%), se construye un mapa de calor de la demanda total sobre Bogota. Esta era una de las motivaciones para elegir este dataset.

In [1]:
import folium
from folium.plugins import HeatMap

geo = pd.concat([
    est.rename(columns={'numero_estacion': 'cod_estacion_raw'})[['cod_estacion_raw','latitud_estacion','longitud_estacion']]
       .assign(cod_estacion=lambda d: d['cod_estacion_raw'].astype(int).astype(str).str.zfill(5))
       [['cod_estacion','latitud_estacion','longitud_estacion']],
    extra.rename(columns={'latitud':'latitud_estacion','longitud':'longitud_estacion'})
         .assign(cod_estacion=lambda d: d['cod_estacion'].astype(int).astype(str).str.zfill(5))
         [['cod_estacion','latitud_estacion','longitud_estacion']],
], ignore_index=True)
geo['latitud_estacion'] = geo['latitud_estacion'].astype(float)
geo['longitud_estacion'] = geo['longitud_estacion'].astype(float)

demanda_estacion = fact.groupby('cod_estacion')['validaciones'].sum().reset_index(name='demanda_total')
demanda_estacion['cod_estacion'] = demanda_estacion['cod_estacion'].astype(str).str.zfill(5)
mapa_df = demanda_estacion.merge(geo, on='cod_estacion', how='inner')

m = folium.Map(location=[mapa_df.latitud_estacion.mean(), mapa_df.longitud_estacion.mean()],
               zoom_start=11, tiles='CartoDB positron')
HeatMap(mapa_df[['latitud_estacion','longitud_estacion','demanda_total']].values.tolist(),
        radius=18, blur=22, max_zoom=13).add_to(m)
m.save('../docs/mapa_calor_demanda.html')
print(f'Mapa guardado, {len(mapa_df)} estaciones geolocalizadas')

Mapa guardado, 140 estaciones geolocalizadas

El archivo interactivo queda en [`docs/mapa_calor_demanda.html`](../docs/mapa_calor_demanda.html) — abrir en el navegador. Sirve tambien como insumo directo para el panel Geomap de Grafana (punto g).

## 2.4 Patrones temporales

Curva horaria promedio (validaciones por hora, promediadas sobre los dias de cada tipo), usando **solo las 2 semanas tipicas** (10-16 agosto y 14-20 septiembre: 12 dias habiles, 2 sabados, 3 domingos).

In [1]:
festivo_semana = [f'2026-08-0{d}' for d in range(3, 10)]
tip = fact[~fact['fecha'].isin(festivo_semana)]
dias_por_tipo = tip.groupby('tipo_dia')['fecha'].nunique()
curva = tip.groupby(['tipo_dia', 'hora'])['validaciones'].sum().unstack('tipo_dia').fillna(0)
curva_prom = curva.div(dias_por_tipo, axis=1)
curva_prom.round(0).astype(int)

tipo_dia  domingo   habil  sabado
hora
6           19486  175691   92734
7           19381  147476   92792
8           21754   98015   78308
...
16          26185  131053   68024
17          27005  156760   65388
18          24245  100939   60747
(24 filas en total, ver notebook; guardado en data/processed/curva_horaria_tipo_dia.csv)

**Lectura:** el dia habil tiene un patron de dos picos muy marcado (6-7am y 16-17h, tipico de commuting laboral). El sabado y el domingo son mucho mas planos, con el maximo cerca del mediodia — un patron social/comercial, no laboral.

### El festivo: ¿se parece mas a un habil o a un fin de semana?

Comparacion dentro de la MISMA semana (3-9 agosto), para que sea justa: habil (lunes a jueves), festivo (viernes 7), sabado y domingo de esa semana.

In [1]:
semana_festivo = fact[fact['fecha'].between('2026-08-03', '2026-08-09')]
dias_sf = semana_festivo.groupby('tipo_dia')['fecha'].nunique()
total_dia_prom = semana_festivo.groupby('tipo_dia')['validaciones'].sum() / dias_sf
total_dia_prom.round(0).astype(int).sort_values(ascending=False)

tipo_dia
habil      1856994
sabado     1062203
festivo     525834
domingo     516447
dtype: int64

**Hallazgo central (responde la pregunta especifica #2 y la hipotesis H3):** el festivo del 7 de agosto tuvo **525,834** validaciones, practicamente identico al domingo de esa misma semana (516,447) y muy por debajo del promedio de un dia habil (1,856,994 — 3.5 veces mas). La forma de la curva horaria del festivo tambien es plana como la de un domingo, sin el pico agudo de las 6-7am del dia habil (ver tabla completa en el notebook). **Conclusion: para efectos operativos, un festivo se debe planear como un domingo, no como un dia habil.**

## 2.5 Concentracion de la demanda entre estaciones

¿Unas pocas estaciones concentran la mayoria de la demanda, o esta repartida parejo?

In [1]:
tot_estacion = tip.groupby('cod_estacion')['validaciones'].sum().sort_values(ascending=False)
nombre = tip.drop_duplicates('cod_estacion').set_index('cod_estacion')['nombre_val']
top15 = tot_estacion.head(15)
print('Top 15 estaciones por demanda total (14 dias tipicos):')
for cod, val in top15.items():
    print(f'  {nombre.get(cod, cod):40s} {val:>10,}')
print(f"\nLas 15 estaciones de mayor demanda concentran el {top15.sum()/tot_estacion.sum()*100:.1f}% del total")

v = np.sort(tot_estacion.values)
n = len(v)
cum = np.cumsum(v) / v.sum()
gini = 1 - 2 * np.trapezoid(cum, np.arange(1, n+1) / n)
print(f"Indice de Gini de concentracion entre las {n} estaciones: {gini:.3f}")

Top 15 estaciones por demanda total (14 dias tipicos):
  Portal Norte - Unicervantes                 864,168
  Portal Americas                             859,425
  Portal Suba                                 680,631
  Portal Sur JFK Coop. Financiera              672,573
  Cabecera Calle 80                             643,959
  Cabecera Usme                                 620,761
  Portal El Dorado - C.C. Nuestro Bogota        577,227
  Portal 20 de Julio                            518,089
  Portal Tunal                                  501,759
  San Mateo - C.C. Unisur                       488,646
  Calle 100 - Marketmedios                      455,930
  Banderas P. Central                           421,478
  Corral Portal Dorado                          374,623
  Terreros                                      329,517
  Toberin                                       303,280

Las 15 estaciones de mayor demanda concentran el 37.8% del total
Indice de Gini de concentracion entre las 154 

**Lectura:** un Gini de 0.48 es una concentracion moderada-alta (0 = todas las estaciones igual de ocupadas, 1 = una sola estacion concentra todo). Tiene sentido: los portales (cabeceras de troncal, con alimentadores) dominan el top 15. Esto ya es evidencia a favor de la hipotesis H1 (la presion se concentra en un subconjunto pequeno de estaciones).

## 2.6 Construccion de features por estacion

Tabla **estacion → features**, la unidad de analisis del clustering (notebook 04). Se calcula solo con las 2 semanas tipicas (sin el festivo, ver `CLAUDE.md`).

In [1]:
dias_habil = tip[tip.tipo_dia == 'habil']['fecha'].nunique()
dias_sab = tip[tip.tipo_dia == 'sabado']['fecha'].nunique()
dias_dom = tip[tip.tipo_dia == 'domingo']['fecha'].nunique()

tot = tip.groupby('cod_estacion')['validaciones'].sum().rename('demanda_total')
prom_habil = tip[tip.tipo_dia=='habil'].groupby('cod_estacion')['validaciones'].sum() / dias_habil
prom_sab = tip[tip.tipo_dia=='sabado'].groupby('cod_estacion')['validaciones'].sum() / dias_sab
prom_dom = tip[tip.tipo_dia=='domingo'].groupby('cod_estacion')['validaciones'].sum() / dias_dom
prom_finde = (prom_sab.add(prom_dom, fill_value=0)) / 2

pico_am = tip[tip.hora.between(5, 9)].groupby('cod_estacion')['validaciones'].sum()
pico_pm = tip[tip.hora.between(16, 20)].groupby('cod_estacion')['validaciones'].sum()

curva_hora = tip.groupby(['cod_estacion', 'hora'])['validaciones'].sum().unstack(fill_value=0)
curva_norm = curva_hora.div(curva_hora.sum(axis=1), axis=0)  # 24 valores que suman 1, por estacion
n_dias_tot = dias_habil + dias_sab + dias_dom
prom_horario = curva_hora.sum(axis=1) / (24 * n_dias_tot)
indice_concentracion = (curva_hora.max(axis=1) / n_dias_tot) / prom_horario

feats = pd.DataFrame({
    'nombre': nombre,
    'demanda_total': tot,
    'pct_pico_am': (pico_am / tot).fillna(0),
    'pct_pico_pm': (pico_pm / tot).fillna(0),
}).fillna(0)
feats['razon_am_pm'] = (feats['pct_pico_am'] / feats['pct_pico_pm'].replace(0, np.nan)).fillna(0)
feats['razon_finde_habil'] = (prom_finde / prom_habil.replace(0, np.nan)).fillna(0)
feats['indice_concentracion_pico'] = indice_concentracion
for h in range(24):
    feats[f'h{h:02d}'] = curva_norm.get(h, 0)

feats = feats.reset_index()
feats.to_parquet(PROC / 'estacion_features.parquet', index=False)
print(feats.shape)
feats[['demanda_total','pct_pico_am','pct_pico_pm','razon_am_pm','razon_finde_habil','indice_concentracion_pico']].describe().round(2)

(154, 30)
       demanda_total  pct_pico_am  pct_pico_pm  razon_am_pm  razon_finde_habil  indice_concentracion_pico
count          154.0        154.0        154.0        154.0              154.0                      154.0
mean        142845.4         0.33         0.36               1.2               0.50                       4.2
std         156831.4         0.21         0.19               1.6               0.22                       2.5
min              1.0         0.00         0.00               0.0               0.00                       2.1
50%          96561.0         0.31         0.35               0.8               0.50                       3.9
max         864168.0         1.00         1.00              10.6               2.50                      24.0

### Arquetipos a simple vista (antes de correr el clustering formal en el notebook 04)

Ordenando por `razon_am_pm` ya aparecen 2 grupos claramente opuestos:

- **Alto AM/PM (se llenan en la manana, ej. Tibanica-Primavera, Terreros, Tintal):** perfil **residencial** — la gente sale de ahi en la manana.
- **Bajo AM/PM (se llenan en la tarde/noche, ej. San Victorino, Museo del Oro, Virrey, Museo Nacional):** perfil de **destino / comercio-cultura-vida nocturna** — la gente llega ahi en la tarde-noche.

Esto es evidencia a favor de la hipotesis H1 (arquetipos claros), y confirma que `razon_am_pm` es una buena feature para el clustering.

## 2.7 Informacion de contexto adicional (complemento al notebook 01)

Con las features ya construidas, se confirma lo anotado en el notebook 01: sin datos de capacidad/frecuencia de buses ni de personal de control actualmente asignado, este analisis mide **presion de demanda**, no saturacion real ni cobertura actual de personal. El `indice_concentracion_pico` (columna arriba) es el insumo directo para responder la pregunta especifica 5 (donde y cuando reforzar personal), que se calcula formalmente en el notebook 04.